In [1]:
import os
import json
import random
from typing import Dict, List, Tuple

import pandas as pd
import numpy as np
from IPython.display import display, HTML

# Set random seeds for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [2]:
from google.colab import auth
from google.cloud import storage

# Authenticate
auth.authenticate_user()
print("GCP authentication successful!")

# Configuration
PROJECT_ID = "msda-llm-finetuning"
PROCESSED_BUCKET = "msda-llm-processed-data"

# Set project
!gcloud config set project {PROJECT_ID} --quiet

# Create local directories
DATA_DIR = "./data"
os.makedirs(f"{DATA_DIR}/processed", exist_ok=True)
os.makedirs(f"{DATA_DIR}/split/theory", exist_ok=True)
os.makedirs(f"{DATA_DIR}/split/code", exist_ok=True)
os.makedirs(f"{DATA_DIR}/split/math", exist_ok=True)

print(f"Project: {PROJECT_ID}")
print(f"Bucket: gs://{PROCESSED_BUCKET}")

GCP authentication successful!
Updated property [core/project].
Project: msda-llm-finetuning
Bucket: gs://msda-llm-processed-data


In [3]:
# Initialize GCS client
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(PROCESSED_BUCKET)

# Files to download
files_to_download = [
    ("theory/theory_train.jsonl", f"{DATA_DIR}/processed/theory_train.jsonl"),
    ("code/code_train.jsonl", f"{DATA_DIR}/processed/code_train.jsonl"),
    ("math/math_train.jsonl", f"{DATA_DIR}/processed/math_train.jsonl"),
]

print("Downloading processed datasets from GCS...\n")

for gcs_path, local_path in files_to_download:
    print(f"   Downloading {gcs_path}...")
    blob = bucket.blob(gcs_path)
    blob.download_to_filename(local_path)
    # Get file size
    file_size = os.path.getsize(local_path) / (1024 * 1024)  # MB
    print(f"      Downloaded ({file_size:.2f} MB)")

print("\nAll processed data downloaded!")

# Verify downloads
print("\nDownloaded files:")
!ls -lh {DATA_DIR}/processed/


      Downloaded (1.04 MB)
      Downloaded (39.86 MB)
      Downloaded (263.19 MB)

All processed data downloaded!

Downloaded files:
total 305M
-rw-r--r-- 1 root root  40M Dec  2 04:03 code_train.jsonl
-rw-r--r-- 1 root root 264M Dec  2 04:03 math_train.jsonl
-rw-r--r-- 1 root root 1.1M Dec  2 04:03 theory_train.jsonl


In [4]:
def load_jsonl(filepath: str) -> List[Dict]:
    """Load JSONL file into a list of dictionaries."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

def save_jsonl(data: List[Dict], filepath: str):
    """Save list of dictionaries to JSONL file."""
    with open(filepath, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"   Saved: {filepath} ({len(data):,} examples)")

def upload_to_gcs(local_path: str, gcs_path: str, bucket_name: str = PROCESSED_BUCKET):
    """Upload file to GCS."""
    blob = bucket.blob(gcs_path)
    blob.upload_from_filename(local_path)
    print(f"    Uploaded: gs://{bucket_name}/{gcs_path}")

def extract_messages(item: Dict) -> Tuple[str, str, str]:
    """Extract system, user, and assistant messages from a conversation item."""
    system_msg = ""
    user_msg = ""
    assistant_msg = ""

    for msg in item.get("messages", []):
        role = msg.get("role", "")
        content = msg.get("content", "")

        if role == "system":
            system_msg = content
        elif role == "user":
            user_msg = content
        elif role == "assistant":
            assistant_msg = content

    return system_msg, user_msg, assistant_msg

def jsonl_to_dataframe(data: List[Dict]) -> pd.DataFrame:
    """Convert JSONL data to a pandas DataFrame for display."""
    rows = []
    for i, item in enumerate(data):
        system_msg, user_msg, assistant_msg = extract_messages(item)
        rows.append({
            "Index": i,
            "User (Question/Prompt)": user_msg[:300] + "..." if len(user_msg) > 300 else user_msg,
            "Assistant (Answer/Response)": assistant_msg[:300] + "..." if len(assistant_msg) > 300 else assistant_msg,
        })
    return pd.DataFrame(rows)

def display_samples(data: List[Dict], n: int = 5, title: str = ""):
    """Display n random samples in a formatted table."""
    if title:
        print(f"\n{'='*80}")
        print(f"{title}")
        print(f"   Total examples: {len(data):,}")
        print('='*80)

    # Get random samples
    sample_indices = random.sample(range(len(data)), min(n, len(data)))
    samples = [data[i] for i in sample_indices]

    # Convert to DataFrame
    df = jsonl_to_dataframe(samples)

    # Display with better formatting
    pd.set_option('display.max_colwidth', 100)
    pd.set_option('display.width', None)

    display(df)

    return df

def train_test_split(data: List[Dict], test_size: float = 0.1, seed: int = 42) -> Tuple[List[Dict], List[Dict]]:
    """
    Split data into training and testing sets.

    Args:
        data: List of data items
        test_size: Fraction of data to use for testing (default 0.1 = 10%)
        seed: Random seed for reproducibility

    Returns:
        Tuple of (train_data, test_data)
    """
    # Shuffle data
    random.seed(seed)
    shuffled_data = data.copy()
    random.shuffle(shuffled_data)

    # Calculate split index
    n_test = int(len(shuffled_data) * test_size)

    # Split
    test_data = shuffled_data[:n_test]
    train_data = shuffled_data[n_test:]

    return train_data, test_data

print("Helper functions defined!")

Helper functions defined!


In [5]:
print("Loading processed datasets...\n")

# Load all datasets
theory_data = load_jsonl(f"{DATA_DIR}/processed/theory_train.jsonl")
code_data = load_jsonl(f"{DATA_DIR}/processed/code_train.jsonl")
math_data = load_jsonl(f"{DATA_DIR}/processed/math_train.jsonl")

print(f"Theory: {len(theory_data):,} examples")
print(f"Code:   {len(code_data):,} examples")
print(f"Math:   {len(math_data):,} examples")
print(f"\nTotal:  {len(theory_data) + len(code_data) + len(math_data):,} examples")

Loading processed datasets...

Theory: 1,657 examples
Code:   19,443 examples
Math:   138,115 examples

Total:  159,215 examples


In [6]:
# Display Theory samples
display_samples(theory_data, n=10, title="THEORY Dataset Samples")


THEORY Dataset Samples
   Total examples: 1,657


,Index,User (Question/Prompt),Assistant (Answer/Response)
0,0,[Linear Regression] If a weight for one variable is higher than for another - can we say that th...,"Yes - if your predictor variables are normalized.\n\nWithout normalization, the weight represent..."
1,1,Explain principal component analysis (PCA) and the types of problems where PCA is applicable.,PCA is used to lower data dimensions by projecting high-dimensional data onto a lower-dimensiona...
2,2,Define dependency parsing in natural language processing.,Dependency parsing is a technique used in NLP to identify the grammatical relationships between ...
3,3,What is the purpose of feature engineering in machine learning?,"Feature engineering involves selecting, transforming, or creating new features from the raw data..."
4,4,Can you explain the concept of overfitting in machine learning?,Overfitting happens when a machine learning model learns not just the signal but also the noise ...
5,5,What is deep learning?,"Deep learning, a branch of machine learning, entails models called neural networks that mimic br..."
6,6,Explain whether machine learning is synonymous with artificial intelligence.,Machine Learning (ML) and Artificial Intelligence (AI) are related concepts but not interchangea...
7,7,"What is a kernel, and explain the kernel trick?",The kernel trick involves mapping data to a higher-dimensional space to resolve non-linear separ...
8,8,What is collaborative filtering?,Collaborative filtering is the most prominent approach to generate recommendations.\n It uses th...
9,9,What are the consequences of setting the learning rate too high or too low?,"Setting the learning rate too low results in slow convergence of the model, whereas a high learn..."


,Index,User (Question/Prompt),Assistant (Answer/Response)
0,0,[Linear Regression] If a weight for one variable is higher than for another - can we say that th...,"Yes - if your predictor variables are normalized.\n\nWithout normalization, the weight represent..."
1,1,Explain principal component analysis (PCA) and the types of problems where PCA is applicable.,PCA is used to lower data dimensions by projecting high-dimensional data onto a lower-dimensiona...
2,2,Define dependency parsing in natural language processing.,Dependency parsing is a technique used in NLP to identify the grammatical relationships between ...
3,3,What is the purpose of feature engineering in machine learning?,"Feature engineering involves selecting, transforming, or creating new features from the raw data..."
4,4,Can you explain the concept of overfitting in machine learning?,Overfitting happens when a machine learning model learns not just the signal but also the noise ...
5,5,What is deep learning?,"Deep learning, a branch of machine learning, entails models called neural networks that mimic br..."
6,6,Explain whether machine learning is synonymous with artificial intelligence.,Machine Learning (ML) and Artificial Intelligence (AI) are related concepts but not interchangea...
7,7,"What is a kernel, and explain the kernel trick?",The kernel trick involves mapping data to a higher-dimensional space to resolve non-linear separ...
8,8,What is collaborative filtering?,Collaborative filtering is the most prominent approach to generate recommendations.\n It uses th...
9,9,What are the consequences of setting the learning rate too high or too low?,"Setting the learning rate too low results in slow convergence of the model, whereas a high learn..."


In [7]:
# Display Code samples
display_samples(code_data, n=10, title="CODE Dataset Samples")


CODE Dataset Samples
   Total examples: 19,443


,Index,User (Question/Prompt),Assistant (Answer/Response)
0,0,Write a SQL query to select all rows in the table 'students' whose marks are greater than 25.,SELECT *\nFROM students\nWHERE marks > 25;
1,1,You are given a dataframe. Generate a bar chart to visually demonstrate the frequencies of each ...,import matplotlib.pyplot as plt\n\nlabels = df['Food'].unique()\ncounts = df['Food'].value_count...
2,2,Construct a matrix in Python containing 5 rows and 4 columns of random floating point numbers be...,"import numpy as np\n\nmatrix = np.random.uniform(0,1, (5,4))"
3,3,"With the Titanic dataset from Kaggle, examine the data types. Write a function in Python using l...","Let's walk this through one step at a time:\n\n1. First, let's load the dataset and examine the ..."
4,4,Use machine learning to predict the number of houses sold for a given month.,#import necessary libraries\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot a...
5,5,Create a machine learning model to classify emails into spam and ham categories.,import numpy as np\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVector...
6,6,Design a neural network with 95% accuracy to classify articles into topics.,# Import libraries\nimport numpy as np\nimport keras\nfrom keras.models import Sequential\nfrom ...
7,7,How can I modify the given Python code to categorize each city in the 'cities_df' dataframe by c...,Here's the modified Python code that should achieve the same result:\n```python\nimport pandas a...
8,8,What methods can we utilize in Python to accurately identify and categorize regional variations ...,To accurately identify and categorize regional variations in a language based on its syntax and ...
9,9,Explore the data_df dataframe and build a visualization that answers the following question: wha...,To visualize the correlation between the number of tweets sent and the sentiment scores for a sp...


,Index,User (Question/Prompt),Assistant (Answer/Response)
0,0,Write a SQL query to select all rows in the table 'students' whose marks are greater than 25.,SELECT *\nFROM students\nWHERE marks > 25;
1,1,You are given a dataframe. Generate a bar chart to visually demonstrate the frequencies of each ...,import matplotlib.pyplot as plt\n\nlabels = df['Food'].unique()\ncounts = df['Food'].value_count...
2,2,Construct a matrix in Python containing 5 rows and 4 columns of random floating point numbers be...,"import numpy as np\n\nmatrix = np.random.uniform(0,1, (5,4))"
3,3,"With the Titanic dataset from Kaggle, examine the data types. Write a function in Python using l...","Let's walk this through one step at a time:\n\n1. First, let's load the dataset and examine the ..."
4,4,Use machine learning to predict the number of houses sold for a given month.,#import necessary libraries\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot a...
5,5,Create a machine learning model to classify emails into spam and ham categories.,import numpy as np\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVector...
6,6,Design a neural network with 95% accuracy to classify articles into topics.,# Import libraries\nimport numpy as np\nimport keras\nfrom keras.models import Sequential\nfrom ...
7,7,How can I modify the given Python code to categorize each city in the 'cities_df' dataframe by c...,Here's the modified Python code that should achieve the same result:\n```python\nimport pandas a...
8,8,What methods can we utilize in Python to accurately identify and categorize regional variations ...,To accurately identify and categorize regional variations in a language based on its syntax and ...
9,9,Explore the data_df dataframe and build a visualization that answers the following question: wha...,To visualize the correlation between the number of tweets sent and the sentiment scores for a sp...


In [8]:
# Display Math samples
display_samples(math_data, n=10, title="MATH Dataset Samples")


MATH Dataset Samples
   Total examples: 138,115


,Index,User (Question/Prompt),Assistant (Answer/Response)
0,0,Determine for how many integers $n$ the expression $(3n + 2i)^6$ is a pure integer?\n- $\text{(A...,1. **Expand the expression**: Use the binomial theorem to expand $(3n + 2i)^6$:\n \[\n (3n + 2i)...
1,1,"The derivative of $f(x)$ at $x$, denoted as $f'(x)=0$, is a condition for $f(x)$ to have an extr...","For example, $f(x)=|x|$ has an extremum at $x=0$, but it is not differentiable at $x=0$, \nso $f..."
2,2,The function $f(x)=x^{3}+3ax^{2}+bx+a^{2}$ has a maximum value of $0$ at $x=-1$. Find the value ...,To find the value of $a+b$ given that the function $f(x)=x^{3}+3ax^{2}+bx+a^{2}$ has a maximum v...
3,3,"Let vectors $\overrightarrow{a}$ and $\overrightarrow{b}$ satisfy $|\overrightarrow{a}|=2$, $|\o...","To solve the problem, let's start by considering the conditions given and what we're trying to f..."
4,4,"From the numbers \\(1, 2, 3, 4, 5\\), two different numbers are randomly selected. Let event \\(...",**Analysis**\n\nThis question examines conditional probability and the probability of mutually e...
5,5,Given that the function $f(x)$ is differentiable on $\mathbb{R}$ and its derivative $f'(x) < \si...,"Let $g(x) = f(x) - \sin 2x$.\nSince $f(x)$ is an odd function, $g(x)$ is also an odd function.\n..."
6,6,"During the Qingming Festival, a certain school, in order to commemorate the revolutionary martyr...",### Step-by-Step Solution\n\n#### Part 1: Finding the values of $a$ and $b$\n\nGiven the ratio o...
7,7,"Through the midpoint of each diagonal of a convex quadrilateral, a line is drawn parallel to the...",1. Consider a convex quadrilateral \(ABCD\) with diagonals \(AC\) and \(BD\).\n2. Let \(M\) and ...
8,8,"I am thinking of several consecutive natural numbers. If we remove the numbers 70, 82, and 103, ...","1. **Establish the Arithmetic Mean Condition**:\n Given that removing the numbers 70, 82, and 10..."
9,9,"Among all constants $M$ such that $f(x) \leq M$ holds true for any $x$, the least value of $M$ i...",We know that\n$$\frac {1}{2a} + \frac {2}{b} = \frac {a+b}{2a} + \frac {2(a+b)}{b} = \frac {5}{2...


,Index,User (Question/Prompt),Assistant (Answer/Response)
0,0,Determine for how many integers $n$ the expression $(3n + 2i)^6$ is a pure integer?\n- $\text{(A...,1. **Expand the expression**: Use the binomial theorem to expand $(3n + 2i)^6$:\n \[\n (3n + 2i)...
1,1,"The derivative of $f(x)$ at $x$, denoted as $f'(x)=0$, is a condition for $f(x)$ to have an extr...","For example, $f(x)=|x|$ has an extremum at $x=0$, but it is not differentiable at $x=0$, \nso $f..."
2,2,The function $f(x)=x^{3}+3ax^{2}+bx+a^{2}$ has a maximum value of $0$ at $x=-1$. Find the value ...,To find the value of $a+b$ given that the function $f(x)=x^{3}+3ax^{2}+bx+a^{2}$ has a maximum v...
3,3,"Let vectors $\overrightarrow{a}$ and $\overrightarrow{b}$ satisfy $|\overrightarrow{a}|=2$, $|\o...","To solve the problem, let's start by considering the conditions given and what we're trying to f..."
4,4,"From the numbers \\(1, 2, 3, 4, 5\\), two different numbers are randomly selected. Let event \\(...",**Analysis**\n\nThis question examines conditional probability and the probability of mutually e...
5,5,Given that the function $f(x)$ is differentiable on $\mathbb{R}$ and its derivative $f'(x) < \si...,"Let $g(x) = f(x) - \sin 2x$.\nSince $f(x)$ is an odd function, $g(x)$ is also an odd function.\n..."
6,6,"During the Qingming Festival, a certain school, in order to commemorate the revolutionary martyr...",### Step-by-Step Solution\n\n#### Part 1: Finding the values of $a$ and $b$\n\nGiven the ratio o...
7,7,"Through the midpoint of each diagonal of a convex quadrilateral, a line is drawn parallel to the...",1. Consider a convex quadrilateral \(ABCD\) with diagonals \(AC\) and \(BD\).\n2. Let \(M\) and ...
8,8,"I am thinking of several consecutive natural numbers. If we remove the numbers 70, 82, and 103, ...","1. **Establish the Arithmetic Mean Condition**:\n Given that removing the numbers 70, 82, and 10..."
9,9,"Among all constants $M$ such that $f(x) \leq M$ holds true for any $x$, the least value of $M$ i...",We know that\n$$\frac {1}{2a} + \frac {2}{b} = \frac {a+b}{2a} + \frac {2(a+b)}{b} = \frac {5}{2...


In [9]:
print("="*80)
print("PERFORMING TRAIN/TEST SPLIT (90% Train / 10% Test)")
print("="*80)

TEST_SIZE = 0.10  # 10% for testing

# Split Theory
theory_train, theory_test = train_test_split(theory_data, test_size=TEST_SIZE, seed=RANDOM_SEED)
print(f"\nTheory:")
print(f"   Original: {len(theory_data):,}")
print(f"   Train:    {len(theory_train):,} ({100 * len(theory_train) / len(theory_data):.1f}%)")
print(f"   Test:     {len(theory_test):,} ({100 * len(theory_test) / len(theory_data):.1f}%)")

# Split Code
code_train, code_test = train_test_split(code_data, test_size=TEST_SIZE, seed=RANDOM_SEED)
print(f"\nCode:")
print(f"   Original: {len(code_data):,}")
print(f"   Train:    {len(code_train):,} ({100 * len(code_train) / len(code_data):.1f}%)")
print(f"   Test:     {len(code_test):,} ({100 * len(code_test) / len(code_data):.1f}%)")

# Split Math
math_train, math_test = train_test_split(math_data, test_size=TEST_SIZE, seed=RANDOM_SEED)
print(f"\nMath:")
print(f"   Original: {len(math_data):,}")
print(f"   Train:    {len(math_train):,} ({100 * len(math_train) / len(math_data):.1f}%)")
print(f"   Test:     {len(math_test):,} ({100 * len(math_test) / len(math_data):.1f}%)")

# Summary
total_train = len(theory_train) + len(code_train) + len(math_train)
total_test = len(theory_test) + len(code_test) + len(math_test)
print(f"\n{'='*80}")
print(f"SPLIT SUMMARY")
print(f"{'='*80}")
print(f"   Total Train: {total_train:,}")
print(f"   Total Test:  {total_test:,}")
print(f"   Total:       {total_train + total_test:,}")

PERFORMING TRAIN/TEST SPLIT (90% Train / 10% Test)

Theory:
   Original: 1,657
   Train:    1,492 (90.0%)
   Test:     165 (10.0%)

Code:
   Original: 19,443
   Train:    17,499 (90.0%)
   Test:     1,944 (10.0%)

Math:
   Original: 138,115
   Train:    124,304 (90.0%)
   Test:     13,811 (10.0%)

SPLIT SUMMARY
   Total Train: 143,295
   Total Test:  15,920
   Total:       159,215


In [10]:
print("="*80)
print("SAVING SPLIT DATASETS LOCALLY")
print("="*80)

# Save Theory
print("\nTheory:")
save_jsonl(theory_train, f"{DATA_DIR}/split/theory/theory_train.jsonl")
save_jsonl(theory_test, f"{DATA_DIR}/split/theory/theory_test.jsonl")

# Save Code
print("\nCode:")
save_jsonl(code_train, f"{DATA_DIR}/split/code/code_train.jsonl")
save_jsonl(code_test, f"{DATA_DIR}/split/code/code_test.jsonl")

# Save Math
print("\nMath:")
save_jsonl(math_train, f"{DATA_DIR}/split/math/math_train.jsonl")
save_jsonl(math_test, f"{DATA_DIR}/split/math/math_test.jsonl")

print("\nAll split datasets saved locally!")

# Verify local files
print("\nLocal files created:")
!ls -lh {DATA_DIR}/split/theory/
!ls -lh {DATA_DIR}/split/code/
!ls -lh {DATA_DIR}/split/math/

SAVING SPLIT DATASETS LOCALLY

Theory:
   Saved: ./data/split/theory/theory_train.jsonl (1,492 examples)
   Saved: ./data/split/theory/theory_test.jsonl (165 examples)

Code:
   Saved: ./data/split/code/code_train.jsonl (17,499 examples)
   Saved: ./data/split/code/code_test.jsonl (1,944 examples)

Math:
   Saved: ./data/split/math/math_train.jsonl (124,304 examples)
   Saved: ./data/split/math/math_test.jsonl (13,811 examples)

All split datasets saved locally!

Local files created:
total 1.1M
-rw-r--r-- 1 root root 101K Dec  2 04:06 theory_test.jsonl
-rw-r--r-- 1 root root 967K Dec  2 04:06 theory_train.jsonl
total 40M
-rw-r--r-- 1 root root 4.0M Dec  2 04:06 code_test.jsonl
-rw-r--r-- 1 root root  36M Dec  2 04:06 code_train.jsonl
total 264M
-rw-r--r-- 1 root root  27M Dec  2 04:06 math_test.jsonl
-rw-r--r-- 1 root root 237M Dec  2 04:06 math_train.jsonl


In [11]:
print("="*80)
print("UPLOADING SPLIT DATASETS TO GCS")
print("="*80)

# Upload Theory
print("\nTheory:")
upload_to_gcs(f"{DATA_DIR}/split/theory/theory_train.jsonl", "theory/theory_train_split.jsonl")
upload_to_gcs(f"{DATA_DIR}/split/theory/theory_test.jsonl", "theory/theory_test.jsonl")

# Upload Code
print("\nCode:")
upload_to_gcs(f"{DATA_DIR}/split/code/code_train.jsonl", "code/code_train_split.jsonl")
upload_to_gcs(f"{DATA_DIR}/split/code/code_test.jsonl", "code/code_test.jsonl")

# Upload Math
print("\nMath:")
upload_to_gcs(f"{DATA_DIR}/split/math/math_train.jsonl", "math/math_train_split.jsonl")
upload_to_gcs(f"{DATA_DIR}/split/math/math_test.jsonl", "math/math_test.jsonl")

print("\nAll split datasets uploaded to GCS!")

UPLOADING SPLIT DATASETS TO GCS

Theory:
    Uploaded: gs://msda-llm-processed-data/theory/theory_train_split.jsonl
    Uploaded: gs://msda-llm-processed-data/theory/theory_test.jsonl

Code:
    Uploaded: gs://msda-llm-processed-data/code/code_train_split.jsonl
    Uploaded: gs://msda-llm-processed-data/code/code_test.jsonl

Math:
    Uploaded: gs://msda-llm-processed-data/math/math_train_split.jsonl
    Uploaded: gs://msda-llm-processed-data/math/math_test.jsonl

All split datasets uploaded to GCS!


In [12]:
print("Verifying GCS uploads:\n")

print("Theory files:")
!gsutil ls -lh gs://{PROCESSED_BUCKET}/theory/

print("\nCode files:")
!gsutil ls -lh gs://{PROCESSED_BUCKET}/code/

print("\nMath files:")
!gsutil ls -lh gs://{PROCESSED_BUCKET}/math/

Verifying GCS uploads:

Theory files:
100.83 KiB  2025-12-02T04:06:28Z  gs://msda-llm-processed-data/theory/theory_test.jsonl
  1.04 MiB  2025-11-29T04:43:31Z  gs://msda-llm-processed-data/theory/theory_train.jsonl
966.89 KiB  2025-12-02T04:06:28Z  gs://msda-llm-processed-data/theory/theory_train_split.jsonl
TOTAL: 3 objects, 2186690 bytes (2.09 MiB)

Code files:
  3.99 MiB  2025-12-02T04:06:29Z  gs://msda-llm-processed-data/code/code_test.jsonl
 39.86 MiB  2025-11-29T04:43:41Z  gs://msda-llm-processed-data/code/code_train.jsonl
 35.87 MiB  2025-12-02T04:06:29Z  gs://msda-llm-processed-data/code/code_train_split.jsonl
TOTAL: 3 objects, 83590264 bytes (79.72 MiB)

Math files:
 26.24 MiB  2025-12-02T04:06:33Z  gs://msda-llm-processed-data/math/math_test.jsonl
263.19 MiB  2025-11-29T04:44:06Z  gs://msda-llm-processed-data/math/math_train.jsonl
236.95 MiB  2025-12-02T04:06:32Z  gs://msda-llm-processed-data/math/math_train_split.jsonl
TOTAL: 3 objects, 551958568 bytes (526.39 MiB)
